In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"

con = duckdb.connect()
con.execute("SET threads=2")
con.execute("SET memory_limit='10GB'")
con.execute("SET max_temp_directory_size='100GB'")
con.execute("SET preserve_insertion_order=false")

con.execute(f"CREATE OR REPLACE VIEW logs AS SELECT * FROM '{DATA_DIR}/logs_all_clean.parquet'")
con.execute(f"CREATE OR REPLACE VIEW train AS SELECT * FROM '{DATA_DIR}/train_v2.parquet'")

# Verify
total = con.execute("SELECT COUNT(*) FROM logs").fetchone()[0]
users = con.execute("SELECT COUNT(DISTINCT msno) FROM logs").fetchone()[0]
print(f"Total log rows: {total:,}")
print(f"Unique users in logs: {users:,}")

Total log rows: 250,470,260
Unique users in logs: 855,160


In [ ]:
schema = con.execute("DESCRIBE logs").df()
print(schema.to_string())

date_range = con.execute("""
    SELECT 
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT date) AS unique_dates
    FROM logs
""").fetchone()
print(f"\nDate range: {date_range[0]} → {date_range[1]}")
print(f"Unique dates: {date_range[2]}")

  column_name column_type null   key default extra
0        msno     VARCHAR  YES  None    None  None
1        date      BIGINT  YES  None    None  None
2      num_25      BIGINT  YES  None    None  None
3      num_50      BIGINT  YES  None    None  None
4      num_75      BIGINT  YES  None    None  None
5     num_985      BIGINT  YES  None    None  None
6     num_100      BIGINT  YES  None    None  None
7     num_unq      BIGINT  YES  None    None  None
8  total_secs      DOUBLE  YES  None    None  None

Date range: 20150101 → 20170331
Unique dates: 821


In [ ]:
logs_features = con.execute(f"""
WITH base AS (
    SELECT
        msno,
        STRPTIME(CAST(date AS VARCHAR), '%Y%m%d')::DATE AS log_date,
        num_25, num_50, num_75, num_985, num_100, num_unq,
        LEAST(total_secs, 86400)                         AS total_secs_capped,
        (num_100 + num_985)                              AS completed,
        (num_25 + num_50 + num_75 + num_985 + num_100)  AS total_songs,
        ROW_NUMBER() OVER (
            PARTITION BY msno ORDER BY date DESC
        )                                                AS rn_desc
    FROM logs
    WHERE msno IN (SELECT msno FROM train)
),
std_30d AS (
    SELECT
        msno,
        STDDEV(total_songs) AS songs_std_30d
    FROM base
    WHERE log_date >= DATE '2017-03-01'
    GROUP BY msno
),
agg AS (
    SELECT
        msno,
        DATE_DIFF('day', MAX(log_date), DATE '2017-03-31')              AS days_since_last_listen,
        MAX(log_date)                                                    AS last_listen_date,
        MIN(log_date)                                                    AS first_listen_date,
        -- Active days per window
        SUM(CASE WHEN log_date >= DATE '2017-03-17' THEN 1 ELSE 0 END)  AS active_days_15d,
        SUM(CASE WHEN log_date >= DATE '2017-03-01' THEN 1 ELSE 0 END)  AS active_days_30d,
        SUM(CASE WHEN log_date >= DATE '2017-01-31' THEN 1 ELSE 0 END)  AS active_days_60d,
        SUM(CASE WHEN log_date >= DATE '2016-12-02' THEN 1 ELSE 0 END)  AS active_days_120d,
        COUNT(*)                                                         AS active_days_all,
        -- Seconds per window
        SUM(CASE WHEN log_date >= DATE '2017-03-17'
                 THEN total_secs_capped ELSE 0 END)                      AS secs_15d,
        SUM(CASE WHEN log_date >= DATE '2017-03-01'
                 THEN total_secs_capped ELSE 0 END)                      AS secs_30d,
        SUM(total_secs_capped)                                           AS secs_all,
        -- Songs per window
        SUM(CASE WHEN log_date >= DATE '2017-03-17'
                 THEN total_songs ELSE 0 END)                            AS songs_15d,
        SUM(CASE WHEN log_date >= DATE '2017-03-01'
                 THEN total_songs ELSE 0 END)                            AS songs_30d,
        SUM(total_songs)                                                 AS songs_all,
        -- Completion lifetime
        SUM(completed)                                                   AS total_completed,
        SUM(total_songs)                                                 AS total_songs_all,
        -- Completion 15d
        SUM(CASE WHEN log_date >= DATE '2017-03-17'
                 THEN completed ELSE 0 END)                              AS completed_15d,
        SUM(CASE WHEN log_date >= DATE '2017-03-17'
                 THEN total_songs ELSE 0 END)                            AS songs_15d_raw,
        -- Unique songs
        SUM(num_unq)                                                     AS unq_songs_all,
        -- Trend windows — no overlap, include Mar 31
        -- curr: Mar 1 to Mar 31 (31 days, full last month)
        -- prev: Feb 1 to Feb 28 (28 days, strictly before Mar 1)
        SUM(CASE WHEN log_date >= DATE '2017-03-01'
                 THEN total_secs_capped ELSE 0 END)                      AS secs_curr_30d,
        SUM(CASE WHEN log_date >= DATE '2017-02-01'
                 AND log_date < DATE '2017-03-01'
                 THEN total_secs_capped ELSE 0 END)                      AS secs_prev_30d,
        SUM(CASE WHEN log_date >= DATE '2017-03-01'
                 THEN 1 ELSE 0 END)                                      AS active_curr_30d,
        SUM(CASE WHEN log_date >= DATE '2017-02-01'
                 AND log_date < DATE '2017-03-01'
                 THEN 1 ELSE 0 END)                                      AS active_prev_30d,
        -- Final day stats via rn_desc — no correlated subquery, no DISTINCT
        MAX(CASE WHEN rn_desc = 1 THEN total_songs END)                  AS final_day_songs,
        MAX(CASE WHEN rn_desc = 1 THEN total_secs_capped END)            AS final_day_secs
    FROM base
    GROUP BY msno
)
SELECT
    a.msno,
    a.days_since_last_listen,
    a.last_listen_date,
    a.first_listen_date,
    -- Active days
    a.active_days_15d,
    a.active_days_30d,
    a.active_days_60d,
    a.active_days_120d,
    a.active_days_all,
    -- Seconds
    a.secs_15d,
    a.secs_30d,
    a.secs_all,
    -- Songs
    a.songs_15d,
    a.songs_30d,
    a.songs_all,
    -- Rates
    CASE WHEN a.total_songs_all > 0
         THEN a.total_completed::DOUBLE / a.total_songs_all
         ELSE NULL END                                                   AS completion_rate_all,
    CASE WHEN a.songs_15d_raw > 0
         THEN a.completed_15d::DOUBLE / a.songs_15d_raw
         ELSE NULL END                                                   AS completion_rate_15d,
    -- Per day
    CASE WHEN a.active_days_all > 0
         THEN a.unq_songs_all::DOUBLE / a.active_days_all
         ELSE NULL END                                                   AS unq_per_day_all,
    CASE WHEN a.active_days_all > 0
         THEN a.songs_all::DOUBLE / a.active_days_all
         ELSE NULL END                                                   AS song_per_day_all,
    -- Trend
    CASE WHEN a.secs_prev_30d > 0
         THEN a.secs_curr_30d::DOUBLE / a.secs_prev_30d
         ELSE NULL END                                                   AS listening_trend_ratio,
    CASE WHEN a.active_prev_30d > 0
         THEN a.active_curr_30d::DOUBLE / a.active_prev_30d
         ELSE NULL END                                                   AS active_days_trend,
    -- Stopped listening
    CASE WHEN a.active_curr_30d = 0 AND a.active_prev_30d > 0
         THEN 1 ELSE 0 END                                               AS stopped_listening,
    -- Tenure
    DATE_DIFF('day', a.first_listen_date, DATE '2017-03-31')            AS listener_tenure_days,
    -- Binary flags
    CASE WHEN a.active_days_15d > 0 THEN 1 ELSE 0 END                  AS has_listening_15d,
    CASE WHEN a.active_days_30d > 0 THEN 1 ELSE 0 END                  AS has_listening_30d,
    -- Final day
    a.final_day_songs,
    a.final_day_secs,
    -- Std dev 30d
    s.songs_std_30d
FROM agg a
LEFT JOIN std_30d s ON a.msno = s.msno
""").df()

print(f"Shape: {logs_features.shape}")
print(f"\nNull counts:\n{logs_features.isnull().sum()[logs_features.isnull().sum() > 0]}")
print(f"\nSample:\n{logs_features[['msno','days_since_last_listen','active_days_15d','active_days_30d','secs_30d','listening_trend_ratio','stopped_listening']].head(5).to_string()}")

Shape: (855160, 28)

Null counts:
completion_rate_15d      143362
listening_trend_ratio     99340
active_days_trend         99340
songs_std_30d            128927
dtype: int64

Sample:
                                           msno  days_since_last_listen  active_days_15d  active_days_30d    secs_30d  listening_trend_ratio  stopped_listening
0  U1RXzmQwdZPkK/dX475IM4LNm27gaB7+jy9Ty+8yrcM=                       4              5.0              6.0   15776.797               1.556275                  0
1  U1bxSuqcrxmzHKi8E4Iu/O/Z0Ma6sIGPq1K9SX+gbb0=                       4              1.0              4.0   12097.478               0.602181                  0
2  U1o7ZWxUS0ZRa5mp/B+iyIFhs5SGDGQfSQJ3ZclpExk=                       0             15.0             31.0  189699.462               0.930687                  0
3  U3V5v6NJZGc2mCXNvs7pOk8qPEtUWbRvOjCYBjfgF5s=                       1             10.0             20.0   97851.272               0.982993                  0
4  U4f9BXxf2kVlN

In [ ]:
patch_logs = con.execute("""
WITH base AS (
    SELECT
        msno,
        num_25,
        (num_25 + num_50 + num_75 + num_985 + num_100) AS total_songs
    FROM logs
    WHERE msno IN (SELECT msno FROM train)
)
SELECT
    msno,
    CASE WHEN SUM(total_songs) > 0
         THEN SUM(num_25)::DOUBLE / SUM(total_songs)
         ELSE NULL END AS skip_rate_all
FROM base
GROUP BY msno
""").df()

print(f"Patch shape: {patch_logs.shape}")
print(f"Nulls: {patch_logs.isnull().sum()}")
print(patch_logs.head(3).to_string())

Patch shape: (855160, 2)
Nulls: msno             0
skip_rate_all    0
dtype: int64
                                           msno  skip_rate_all
0  yxiEWwE9VR5utpUecLxVdQ5B7NysUPfrNtGINaM2zA8=       0.082824
1  q3MXOPoaa2SCN4bnPQ0Jr7o4vuN/F0FVhSLZufI70SM=       0.232577
2  GI65XroKbX7GEywRnPDVL7xc6ZTy/yGstLrYcIdrH0U=       0.147926


In [ ]:
patch2_logs = con.execute("""
WITH base AS (
    SELECT
        msno,
        STRPTIME(CAST(date AS VARCHAR), '%Y%m%d')::DATE AS log_date,
        num_25, num_100, num_unq,
        LEAST(total_secs, 86400)                         AS total_secs_capped,
        (num_25 + num_50 + num_75 + num_985 + num_100)  AS total_songs,
        (num_100 + num_985)                              AS completed,
        ROW_NUMBER() OVER (
            PARTITION BY msno ORDER BY date DESC
        )                                                AS rn_desc
    FROM logs
    WHERE msno IN (SELECT msno FROM train)
)
SELECT
    msno,
    -- windowed raw counts
    SUM(CASE WHEN log_date >= DATE '2017-03-17'
             THEN num_unq ELSE 0 END)                        AS num_unq_15d,
    SUM(CASE WHEN log_date >= DATE '2017-03-01'
             THEN num_unq ELSE 0 END)                        AS num_unq_30d,
    SUM(CASE WHEN log_date >= DATE '2017-03-17'
             THEN num_100 ELSE 0 END)                        AS num_100_15d,
    SUM(CASE WHEN log_date >= DATE '2017-03-01'
             THEN num_100 ELSE 0 END)                        AS num_100_30d,
    SUM(CASE WHEN log_date >= DATE '2017-03-17'
             THEN num_25 ELSE 0 END)                         AS num_25_15d,
    SUM(CASE WHEN log_date >= DATE '2017-03-01'
             THEN num_25 ELSE 0 END)                         AS num_25_30d,
    -- prev 15d: Mar 1-16 (aligns with secs_30d start)
    SUM(CASE WHEN log_date >= DATE '2017-03-01'
             AND log_date < DATE '2017-03-17'
             THEN 1 ELSE 0 END)                              AS prev_active_days_15d,
    SUM(CASE WHEN log_date >= DATE '2017-03-01'
             AND log_date < DATE '2017-03-17'
             THEN total_secs_capped ELSE 0 END)              AS prev_secs_15d,
    SUM(CASE WHEN log_date >= DATE '2017-03-01'
             AND log_date < DATE '2017-03-17'
             THEN total_songs ELSE 0 END)                    AS prev_songs_15d,
    -- final day rates
    MAX(CASE WHEN rn_desc = 1
             THEN CASE WHEN total_songs > 0
                       THEN completed::DOUBLE / total_songs
                       ELSE NULL END
             END)                                            AS final_day_completion_rate,
    MAX(CASE WHEN rn_desc = 1
             THEN CASE WHEN total_songs > 0
                       THEN num_25::DOUBLE / total_songs
                       ELSE NULL END
             END)                                            AS final_day_skip_rate,
    -- std devs
    STDDEV(CASE WHEN log_date >= DATE '2017-03-17'
                THEN total_songs ELSE NULL END)              AS songs_std_15d,
    STDDEV(CASE WHEN log_date >= DATE '2017-03-01'
                THEN total_secs_capped ELSE NULL END)        AS secs_std_30d,
    STDDEV(CASE WHEN log_date >= DATE '2017-03-17'
                THEN total_secs_capped ELSE NULL END)        AS secs_std_15d
FROM base
GROUP BY msno
""").df()

print(f"Patch2 shape: {patch2_logs.shape}")
print(f"\nNulls:\n{patch2_logs.isnull().sum()[patch2_logs.isnull().sum() > 0]}")
print(patch2_logs.head(3).to_string())

Patch2 shape: (855160, 15)

Nulls:
songs_std_15d    187807
secs_std_30d     128927
secs_std_15d     187807
dtype: int64
                                           msno  num_unq_15d  num_unq_30d  num_100_15d  num_100_30d  num_25_15d  num_25_30d  prev_active_days_15d  prev_secs_15d  prev_songs_15d  final_day_completion_rate  final_day_skip_rate  songs_std_15d  secs_std_30d  secs_std_15d
0  ++mdQeVStSHeEN+/Cfg28lCB7I9zRuJEqFc3AM2gArA=         77.0        120.0         76.0        113.0         2.0         5.0                   3.0      10604.808            48.0                        1.0                  0.0      14.317821   2393.817711   2726.837427
1  +1kXppBPHxRLpqYOHuiareditfCUNeWDjjgT3IfFjmk=          0.0          8.0          0.0         20.0         0.0         1.0                   4.0       5082.321            26.0                        1.0                  0.0            NaN    560.385993           NaN
2  +1p6C1nzhS1+oPhyGOZXCQS1XQxNHB9q4vciNLd9NOk=        353.0        769.0   

In [ ]:
# song per day windowed
logs_features['song_per_day_30d'] = (
    logs_features['songs_30d'] / logs_features['active_days_30d'].replace(0, np.nan)
)
logs_features['song_per_day_15d'] = (
    logs_features['songs_15d'] / logs_features['active_days_15d'].replace(0, np.nan)
)

# merge patch2 only — skip_rate_all already in logs_features from Cell 4
logs_features = logs_features.merge(patch2_logs, on='msno', how='left')

print(f"Final shape: {logs_features.shape}")
print(f"\nNulls (non-zero only):\n{logs_features.isnull().sum()[logs_features.isnull().sum() > 0]}")

out_path = DATA_DIR / "logs_features.parquet"
logs_features.to_parquet(out_path, index=False)

import os
size_mb = os.path.getsize(out_path) / 1024**2
print(f"Saved: {out_path}")
print(f"Size: {size_mb:.1f} MB")
print(f"Shape verify: {pd.read_parquet(out_path).shape}")

Final shape: (855160, 44)

Nulls (non-zero only):
completion_rate_15d      143362
listening_trend_ratio     99340
active_days_trend         99340
songs_std_30d            128927
song_per_day_30d         100609
song_per_day_15d         143362
songs_std_15d            187807
secs_std_30d             128927
secs_std_15d             187807
dtype: int64
Saved: /Users/harshithnr/kkbox-churn/data/parquet/logs_features.parquet
Size: 140.0 MB
Shape verify: (855160, 44)


In [12]:
print(list(logs_features.columns))

['msno', 'days_since_last_listen', 'last_listen_date', 'first_listen_date', 'active_days_15d', 'active_days_30d', 'active_days_60d', 'active_days_120d', 'active_days_all', 'secs_15d', 'secs_30d', 'secs_all', 'songs_15d', 'songs_30d', 'songs_all', 'completion_rate_all', 'completion_rate_15d', 'unq_per_day_all', 'song_per_day_all', 'listening_trend_ratio', 'active_days_trend', 'stopped_listening', 'listener_tenure_days', 'has_listening_15d', 'has_listening_30d', 'final_day_songs', 'final_day_secs', 'songs_std_30d', 'song_per_day_30d', 'song_per_day_15d', 'num_unq_15d', 'num_unq_30d', 'num_100_15d', 'num_100_30d', 'num_25_15d', 'num_25_30d', 'prev_active_days_15d', 'prev_secs_15d', 'prev_songs_15d', 'final_day_completion_rate', 'final_day_skip_rate', 'songs_std_15d', 'secs_std_30d', 'secs_std_15d']


In [13]:
print(patch_logs.shape)

(855160, 2)


In [ ]:
logs_features = logs_features.merge(patch_logs, on='msno', how='left')

print(f"Shape: {logs_features.shape}")
print(f"skip_rate_all nulls: {logs_features['skip_rate_all'].isnull().sum()}")

out_path = DATA_DIR / "logs_features.parquet"
logs_features.to_parquet(out_path, index=False)

import os
size_mb = os.path.getsize(out_path) / 1024**2
print(f"Saved: {out_path}")
print(f"Shape verify: {pd.read_parquet(out_path).shape}")

Shape: (855160, 45)
skip_rate_all nulls: 0
Saved: /Users/harshithnr/kkbox-churn/data/parquet/logs_features.parquet
Shape verify: (855160, 45)
